# 安全意识合成问答数据集生成器（CSV）

## 练习目标（理念）

用本地 **Llama**（4-bit 量化）**合成**一批可公开使用的安全意识培训问答数据，供后续 **RAG（Retrieval-Augmented Generation）** 或内部助手当训练/评测材料。

每次点击生成 **一条**结构化样例：
- **CONTEXT**：短篇教学小手册（约 120–180 词）
- **QUESTION**：只能根据 CONTEXT 回答的问题
- **ANSWER**：只基于 CONTEXT 的 grounded 答案（不引入外部知识）

落盘为 CSV，字段包括：
- `chunk_id`（递增 ID，如 `SA-000001`）
- `topic`、`difficulty`、`tags`
- `context`、`question`、`answer`

## 和本课 Week 3 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 本地开源 LLM | `meta-llama/Llama-3.2-3B-Instruct` |
| 4-bit 量化（QLoRA 相关栈） | `BitsAndBytesConfig` + `device_map="auto"` |
| 聊天模板 | `tokenizer.apply_chat_template(...)` |
| 合成数据 / 结构化输出 | 固定分隔符 `===CONTEXT===` 等 |
| Gradio UI | 一键生成 + 追加写入 CSV |

## 怎么跑

1. 在 Colab（或等价环境）从上到下运行单元格
2. 在 Colab Secrets 配置 `HF_TOKEN`（需有权下载 Llama）
3. 建议有 GPU；无 GPU 也能跑但更慢
4. 打开 Gradio：选主题 → Generate → 检查预览 → Save 追加到 CSV


In [ ]:
# ========== 安装依赖：量化推理 + Gradio UI ==========

# accelerate：支持 device_map="auto"，自动把层放到 GPU/CPU
# bitsandbytes：4-bit 量化（Quantization），显著省显存
# gradio：搭交互界面，点按钮生成/保存样例
!pip install -q --upgrade  accelerate bitsandbytes gradio


In [ ]:
# ========== 导入：标准库 + PyTorch + Transformers + Colab + Gradio ==========

# 标准库：路径/环境变量、JSON、UUID、时间戳
import os                      # File paths, environment variables（路径与环境变量）
import json                    # JSON serialization（序列化）
import uuid                    # Unique IDs for dataset entries（唯一 ID，本练习主用自增 chunk_id）
from datetime import datetime  # Timestamps for filenames/logging（时间戳）

# PyTorch：张量与 GPU 检测
import torch                 # Tensors, GPU check

# Hugging Face / Transformers：登录、分词器、因果语言模型、量化配置
from huggingface_hub import login                 # HF login to download gated models（门禁模型需登录）
from transformers import (
    AutoTokenizer,                                 # Tokenizer loader（分词器）
    AutoModelForCausalLM,                           # Causal LM loader（因果语言模型）
    BitsAndBytesConfig                              # 4-bit quant config（4-bit 量化配置）
)

# Colab Secrets：从密钥库读 HF_TOKEN，避免写进笔记本
from google.colab import userdata                  # Colab Secrets store

# Gradio：后面搭「生成 / 保存」UI
import gradio as gr                                # Gradio UI


In [ ]:
# ========== 模型名 + HF 登录 + GPU 检查 ==========

# Llama 3.2 3B Instruct：门禁模型，需 HF 账号授权后才能下载
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

# 从 Colab「Secrets」读取 Hugging Face token（Environment Variables / Secrets）
hf_token = userdata.get("HF_TOKEN")

# 登录 Hugging Face，才能拉取 Llama 权重
login(hf_token, add_to_git_credential=True)

# 检查是否有 CUDA GPU（推荐：推理更快）
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    # 打印当前 GPU 名称，便于确认运行环境
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ========== 4-bit 量化配置 + 加载分词器与模型 ==========

# BitsAndBytesConfig：把权重量化到 4-bit，降低显存占用
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 启用 4-bit 权重加载
    bnb_4bit_use_double_quant=True,         # 双重量化，进一步压缩
    bnb_4bit_compute_dtype=torch.bfloat16,  # 矩阵乘计算用 bfloat16
    bnb_4bit_quant_type="nf4"               # NF4 量化类型（NormalFloat 4-bit）
)

# 按模型名加载对应 tokenizer（chat template 也在这里）
tokenizer = AutoTokenizer.from_pretrained(LLAMA)

# 部分 Llama tokenizer 没有 pad_token；用 eos 顶上，避免 padding 报错
tokenizer.pad_token = tokenizer.eos_token

# device_map="auto"：按可用设备自动放置各层；挂上 4-bit 量化配置
model = AutoModelForCausalLM.from_pretrained(
    LLAMA,
    device_map="auto",                      # 自动把层放到 GPU/CPU
    quantization_config=quant_config        # 应用上面的 4-bit 配置
)

# 确认模型已加载（推理前可再 model.eval()；此处保持原逻辑不改）

print("Model loaded:", LLAMA)


In [ ]:
# ========== 主题词表 TOPICS：键=展示名，值=生成要求（英文，影响模型行为） ==========

# 每个主题对应一段英文 Topic requirements，发给模型当约束（勿翻译，改译会改变生成内容）
TOPICS = {
    "Phishing Emails — Red Flags": (
        "Explain common phishing indicators (sender spoofing, urgency, suspicious links/attachments, "
        "credential requests). Describe what a user should do if they suspect phishing."
    ),
    "Strong Passwords and Passphrases": (
        "Explain why unique passwords matter, what makes a password strong, and how passphrases work. "
        "Include practical guidance and common mistakes to avoid."
    ),
    "Multi-Factor Authentication (MFA) Basics": (
        "Explain what MFA is, why it helps, and common MFA methods. Mention 'push fatigue' and the "
        "importance of verifying login prompts."
    ),
    "Software Updates and Patching": (
        "Explain why updates matter, how vulnerabilities are exploited, and safe habits for updating "
        "operating systems and apps."
    ),
    "Safe Use of Public Wi-Fi": (
        "Explain risks of public Wi-Fi, what information should not be entered, and safer alternatives "
        "like mobile hotspots or VPN usage (conceptually, no vendor names)."
    ),
    "Handling Sensitive Data": (
        "Explain basic data sensitivity levels (public/internal/confidential), secure sharing habits, "
        "and how to reduce accidental exposure."
    ),
    "Social Engineering by Phone": (
        "Explain pretexting, why verification is necessary, and safe steps to validate identity before "
        "sharing information or granting access."
    )
}

# 打印全部主题名，确认词表加载成功
print("Topics:")
for k in TOPICS.keys():
    print("-", k)


In [ ]:
# ========== 核心：主题→tags + generate_one_text（生成一条 CONTEXT/Q/A） ==========

def _topic_to_tags(topic_name: str) -> str:
    """
    根据主题名推导短标签（tags）。
    写入 CSV 时用 '|' 拼成单个字符串。
    """
    # 小写化便于子串匹配
    base = topic_name.lower()
    # 所有样例都带安全意识总标签
    tags = ["security_awareness"]

    # 按关键词追加领域标签（phishing / password / MFA 等）
    if "phishing" in base:
        tags += ["phishing", "email", "links"]
    if "password" in base or "passphrase" in base:
        tags += ["passwords", "passphrases", "credentials"]
    if "mfa" in base or "multi-factor" in base:
        tags += ["mfa", "authentication", "push_fatigue"]
    if "update" in base or "patch" in base:
        tags += ["updates", "patching", "vulnerabilities"]
    if "wi" in base or "wifi" in base:
        tags += ["public_wifi", "network", "privacy"]
    if "sensitive data" in base or "handling sensitive" in base:
        tags += ["sensitive_data", "sharing", "classification"]
    if "social engineering" in base or "phone" in base:
        tags += ["social_engineering", "verification", "pretexting"]

    # 去重且保序，最多保留 6 个
    dedup = []
    for t in tags:
        if t not in dedup:
            dedup.append(t)

    return "|".join(dedup[:6])


def generate_one_text(topic_name: str, difficulty: str, temperature: float = 0.2, max_new_tokens: int = 450) -> dict:
    """
    用固定分隔符生成「一条」英文样例。
    返回：topic, difficulty, tags, context, question, answer

    关键点：
    - 只 decode 新生成的 token，不要把 prompt 一并 decode 进去。
    """

    # 取出该主题的英文要求；并生成 tags
    topic_desc = TOPICS[topic_name]
    tags = _topic_to_tags(topic_name)

    # system prompt 保留英文：发给模型的指令，改译会改变行为
    system_message = (
        "You are a technical instructor for security awareness (non-expert audience). "
        "You must write a didactic mini-manual chunk and a grounded Q/A pair. "
        "Use EXACTLY the separators requested and fill EVERY section. "
        "Do not mention vendor or brand names."
    )

    # user prompt：主题 + 难度 + 强制分隔符结构（英文规则影响解析）
    user_prompt = f"""
Topic: {topic_name}
Topic requirements: {topic_desc}
Difficulty: {difficulty}

Return output in ENGLISH using EXACTLY this structure:

===CONTEXT===
Write the context here.

===QUESTION===
Write the question here.

===ANSWER===
Write the answer here.

Rules:
- CONTEXT must be 120 to 180 words.
- QUESTION must be answerable ONLY using the CONTEXT.
- ANSWER must use ONLY the CONTEXT (no outside knowledge).
- Do not include any extra sections or commentary.
- Do not leave any section empty.
""".strip()

    # Chat Completions 风格 messages：system 定角色，user 放任务
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt},
    ]

    # 套用 Llama chat template，得到模型可吃的 token 序列
    tokenized = tokenizer.apply_chat_template(messages, return_tensors="pt")

    # 兼容返回 Tensor 或 BatchEncoding 两种形态
    if isinstance(tokenized, torch.Tensor):
        input_ids = tokenized
        attention_mask = None
    else:
        input_ids = tokenized["input_ids"]
        attention_mask = tokenized.get("attention_mask", None)

    # 有 GPU 就把输入搬到 cuda
    if torch.cuda.is_available():
        input_ids = input_ids.to("cuda")
        if attention_mask is not None:
            attention_mask = attention_mask.to("cuda")

    # 记录 prompt 长度，后面只 decode 新增部分
    prompt_len = input_ids.shape[1]

    # 推理：关闭梯度，节省显存
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=float(temperature),
            pad_token_id=tokenizer.eos_token_id
        )

    # 只解码新生成的 token（去掉 prompt 前缀）
    new_tokens = output_ids[0, prompt_len:]
    decoded_new = tokenizer.decode(new_tokens, skip_special_tokens=True)

    # 小工具：截取分隔符 a 与 b 之间的文本
    def _between(text, a, b):
        i = text.find(a)
        if i == -1:
            return ""
        i += len(a)
        j = text.find(b, i)
        if j == -1:
            return text[i:].strip()
        return text[i:j].strip()

    # 按固定分隔符切出三块
    context = _between(decoded_new, "===CONTEXT===", "===QUESTION===")
    question = _between(decoded_new, "===QUESTION===", "===ANSWER===")
    answer = decoded_new.split("===ANSWER===")[-1].strip() if "===ANSWER===" in decoded_new else ""

    # Fallback：分隔符全失败时，把原文放进 answer，保证 UI 有东西看
    if not context and not question and not answer:
        return {
            "topic": topic_name,
            "difficulty": difficulty,
            "tags": tags,
            "context": "",
            "question": "",
            "answer": decoded_new.strip(),
        }

    # 正常路径：返回结构化字典
    return {
        "topic": topic_name,
        "difficulty": difficulty,
        "tags": tags,
        "context": context,
        "question": question,
        "answer": answer,
    }

# 快速冒烟测试：生成一条钓鱼邮件主题样例并打印 tags / question
ex = generate_one_text("Phishing Emails — Red Flags", "medium", temperature=0.2)
print(ex["tags"])
print(ex["question"])


In [ ]:
# ========== CSV 落盘：自增 chunk_id + 追加一行 + 预览 ==========

import csv
import os

# Colab 上的默认 CSV 路径
CSV_PATH = "/content/security_awareness_qa.csv"

# CSV 列顺序（DictWriter 的 fieldnames）
CSV_COLUMNS = ["chunk_id", "topic", "difficulty", "tags", "context", "question", "answer"]

def _next_chunk_id(path: str = CSV_PATH) -> str:
    """
    生成递增 ID：SA-000001, SA-000002, ...
    依据：已有数据行数（去掉表头）。
    """
    # 文件不存在 → 从 1 号开始
    if not os.path.exists(path):
        return "SA-000001"

    # 统计总行数（含表头）
    with open(path, "r", encoding="utf-8") as f:
        n_lines = sum(1 for _ in f)

    # 数据行 = 总行 - 1（表头）；下一号 = 数据行 + 1
    n_data = max(0, n_lines - 1)  # subtract header
    next_id = n_data + 1
    return f"SA-{next_id:06d}"

def append_example_to_csv(example: dict, path: str = CSV_PATH) -> str:
    """
    把一条样例追加进 CSV，并自动分配 chunk_id。
    """
    # 是否已有文件：决定要不要写表头
    file_exists = os.path.exists(path)
    chunk_id = _next_chunk_id(path)

    # 组装一行；缺字段用空串兜底
    row = {
        "chunk_id": chunk_id,
        "topic": example.get("topic", ""),
        "difficulty": example.get("difficulty", ""),
        "tags": example.get("tags", ""),
        "context": example.get("context", ""),
        "question": example.get("question", ""),
        "answer": example.get("answer", ""),
    }

    # 追加模式打开；utf-8 + newline="" 是 csv 模块推荐写法
    with open(path, "a", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_COLUMNS)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)

    return path

def preview_csv(path: str = CSV_PATH, n_lines: int = 5) -> str:
    """
    预览 CSV 前几行（给人看的文本）。
    """
    if not os.path.exists(path):
        return "(CSV not created yet.)"

    lines = []
    with open(path, "r", encoding="utf-8") as f:
        for _ in range(n_lines):
            line = f.readline()
            if not line:
                break
            lines.append(line.rstrip("\n"))
    return "\n".join(lines)

# 可选重置：需要清空数据集时再取消注释（默认注释掉，避免误删）
# if os.path.exists(CSV_PATH): os.remove(CSV_PATH)
# print("Reset:", CSV_PATH)


In [ ]:
# ========== Gradio UI：生成一条 → 预览 → 保存到 CSV ==========

import gradio as gr
import traceback

# 记住「上一次生成」的样例，供 Save 按钮使用
_last_example = None

def ui_generate(topic_name, difficulty, temperature):
    """UI 回调：生成一条样例，返回预览文本 + 状态。"""
    global _last_example
    try:
        # 调用核心生成函数；temperature 转成 float
        ex = generate_one_text(topic_name, difficulty, temperature=float(temperature))
        _last_example = ex

        # 拼预览：主题/难度/标签 + 三段内容（分隔符与生成约定一致）
        preview = (
            f"TOPIC: {ex['topic']}\n"
            f"DIFFICULTY: {ex['difficulty']}\n"
            f"TAGS: {ex['tags']}\n\n"
            f"===CONTEXT===\n{ex['context']}\n\n"
            f"===QUESTION===\n{ex['question']}\n\n"
            f"===ANSWER===\n{ex['answer']}\n"
        )
        return preview, "Generated OK (not saved yet)."
    except Exception:
        # 失败时清空缓存，并把完整 traceback 显示到输出框
        _last_example = None
        return traceback.format_exc(), "Generation failed (see error above)."

def ui_save():
    """UI 回调：把上次生成的样例追加写入 CSV。"""
    global _last_example
    try:
        if _last_example is None:
            return "Nothing to save. Generate first."

        # 追加一行，并返回路径 + CSV 前几行预览
        append_example_to_csv(_last_example, CSV_PATH)
        return f"Saved to: {CSV_PATH}\n\nPreview:\n{preview_csv(CSV_PATH, n_lines=5)}"
    except Exception:
        return traceback.format_exc()

# Blocks：自由排布控件（比 Interface 更灵活）
with gr.Blocks() as demo:
    # 标题与用法说明（Gradio 展示字符串，保留英文 UI 文案以免改变界面行为）
    gr.Markdown("# Security Awareness Q/A — CSV Dataset Generator")
    gr.Markdown("Generate 1 example (context + question + answer) and append it to a CSV with chunk_id and tags.")

    # 同一行：主题 / 难度 / temperature
    with gr.Row():
        topic = gr.Dropdown(list(TOPICS.keys()), value="Phishing Emails — Red Flags", label="Topic")
        difficulty = gr.Dropdown(["easy", "medium", "hard"], value="medium", label="Difficulty")
        temperature = gr.Slider(0.1, 1.0, step=0.1, value=0.2, label="Temperature (lower = more stable)")

    # 生成按钮 + 输出区 + 状态行
    gen_btn = gr.Button("Generate 1 example")
    out_text = gr.Textbox(label="Output / Errors", lines=18)
    status = gr.Textbox(label="Status", lines=1)

    # 保存按钮 + 保存状态（含 CSV 预览）
    save_btn = gr.Button("Save last row to CSV")
    save_status = gr.Textbox(label="Save status", lines=8)

    # 绑定点击事件：生成 → out_text/status；保存 → save_status
    gen_btn.click(ui_generate, inputs=[topic, difficulty, temperature], outputs=[out_text, status])
    save_btn.click(ui_save, inputs=None, outputs=[save_status])

# Colab 里 share=True 通常最稳；prevent_thread_lock 避免阻塞后续单元格
demo.launch(share=True, debug=True, prevent_thread_lock=True)
